## Updating the CSV files to include the correct paths

In [29]:
import numpy as np
import pandas as pd
import os

In [30]:
df = pd.read_csv("calc_case_description_train_set.csv")

# Columns that contain paths
path_cols = [
    "image file path",
    "cropped image file path",
    "ROI mask file path"
]

def extract_uid_path(path):
    if pd.isna(path):
        return path
    
    parts = str(path).split("/")
    
    # Keep only the last two UID folders (ignore filename if present)
    # Example:
    # [..., UID1, UID2, 000000.dcm] → UID1/UID2
    if parts[-1].endswith(".dcm"):
        return "/".join(parts[-3:-1])
    else:
        return "/".join(parts[-2:])

# Apply transformation
for col in path_cols:
    if col in df.columns:
        df[col] = df[col].apply(extract_uid_path)
df.head()


,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,1.3.6.1.4.1.9590.100.1.2.408909860712120272633...,1.3.6.1.4.1.9590.100.1.2.393344010211719049419...,1.3.6.1.4.1.9590.100.1.2.328778919012412769218...
1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,1.3.6.1.4.1.9590.100.1.2.427517897311902339923...,1.3.6.1.4.1.9590.100.1.2.296281207812130400303...,1.3.6.1.4.1.9590.100.1.2.675123622103196361081...
2,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,1.3.6.1.4.1.9590.100.1.2.201322325113694962619...,1.3.6.1.4.1.9590.100.1.2.314135871111943890422...,1.3.6.1.4.1.9590.100.1.2.241202057913673145232...
3,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,1.3.6.1.4.1.9590.100.1.2.370479499712916693322...,1.3.6.1.4.1.9590.100.1.2.914582796124855152034...,1.3.6.1.4.1.9590.100.1.2.314250272911170289203...
4,P_00008,1,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3,1.3.6.1.4.1.9590.100.1.2.162256682111885666305...,1.3.6.1.4.1.9590.100.1.2.281397494612871934937...,1.3.6.1.4.1.9590.100.1.2.336811694512764490002...


In [31]:
print(df.loc[0, "image file path"])
print(df.loc[0, "cropped image file path"])

1.3.6.1.4.1.9590.100.1.2.408909860712120272633130274602115723157/1.3.6.1.4.1.9590.100.1.2.47414316010368386519740343172775938548
1.3.6.1.4.1.9590.100.1.2.393344010211719049419601138200355094682/000001.dcm



In [32]:
BASE_DIR = "../data"

def check_which_uid(uid_path):
    if pd.isna(uid_path):
        return None, None, "missing"
    
    parts = str(uid_path).split("/")
    if len(parts) < 2:
        return None, None, "invalid_format"
    
    uid1, uid2 = parts[0], parts[1]
    
    path1 = os.path.join(BASE_DIR, uid1)
    path2 = os.path.join(BASE_DIR, uid2)
    
    exists1 = os.path.isdir(path1)
    exists2 = os.path.isdir(path2)
    
    if exists1 and exists2:
        return path1, path2, "both"
    elif exists1:
        return path1, None, "uid1"
    elif exists2:
        return None, path2, "uid2"
    else:
        return None, None, "neither"

# Apply
results = df["image file path"].apply(check_which_uid)

df[["uid1_path", "uid2_path", "which_exists"]] = pd.DataFrame(results.tolist(), index=df.index)

# --- Summary ---
print(df["which_exists"].value_counts())

# --- Peek at examples ---
print("\nExamples:")
print(df[["image file path", "which_exists"]].head(10))

which_exists
uid2    1546
Name: count, dtype: int64

Examples:
                                     image file path which_exists
0  1.3.6.1.4.1.9590.100.1.2.408909860712120272633...         uid2
1  1.3.6.1.4.1.9590.100.1.2.427517897311902339923...         uid2
2  1.3.6.1.4.1.9590.100.1.2.201322325113694962619...         uid2
3  1.3.6.1.4.1.9590.100.1.2.370479499712916693322...         uid2
4  1.3.6.1.4.1.9590.100.1.2.162256682111885666305...         uid2
5  1.3.6.1.4.1.9590.100.1.2.162256682111885666305...         uid2
6  1.3.6.1.4.1.9590.100.1.2.162256682111885666305...         uid2
7  1.3.6.1.4.1.9590.100.1.2.841558427127656604294...         uid2
8  1.3.6.1.4.1.9590.100.1.2.841558427127656604294...         uid2
9  1.3.6.1.4.1.9590.100.1.2.841558427127656604294...         uid2


In [ ]:
# --- Step 1: Keep only UID2 ---
def keep_uid2(uid_path):
    if pd.isna(uid_path):
        return uid_path
    
    parts = str(uid_path).split("/")
    
    if len(parts) < 2:
        return uid_path  # leave unchanged if malformed
    
    return parts[1]  # keep UID2 only
def keep_uid1(uid_path):
    if pd.isna(uid_path):
        return uid_path
    
    parts = str(uid_path).split("/")
    
    if len(parts) < 2:
        return uid_path  # leave unchanged if malformed
    
    return parts[0]  # keep UID1 only

path2_cols = [
    "image file path",
    "ROI mask file path"
]
path1_col = ["cropped image file path"]


for col in path2_cols:
    if col in df.columns:
        df[col] = df[col].apply(keep_uid2)

for col in path1_col:
    if col in df.columns:
        df[col] = df[col].apply(keep_uid1)

# --- Step 2: Drop unwanted columns ---
cols_to_drop = [
    "uid1_path_found",
    "uid2_path_found",
    "exists_in_data",
    "uid1_path",
    "uid2_path",
    "which_exists"
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

df.head()


,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety,image file path,cropped image file path,ROI mask file path
0,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,1.3.6.1.4.1.9590.100.1.2.474143160103683865197...,1.3.6.1.4.1.9590.100.1.2.393344010211719049419...,1.3.6.1.4.1.9590.100.1.2.393344010211719049419...
1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3,1.3.6.1.4.1.9590.100.1.2.250596608311207922527...,1.3.6.1.4.1.9590.100.1.2.296281207812130400303...,1.3.6.1.4.1.9590.100.1.2.296281207812130400303...
2,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,1.3.6.1.4.1.9590.100.1.2.228699627313487111012...,1.3.6.1.4.1.9590.100.1.2.314135871111943890422...,1.3.6.1.4.1.9590.100.1.2.314135871111943890422...
3,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4,1.3.6.1.4.1.9590.100.1.2.104743410411133110629...,1.3.6.1.4.1.9590.100.1.2.914582796124855152034...,1.3.6.1.4.1.9590.100.1.2.914582796124855152034...
4,P_00008,1,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3,1.3.6.1.4.1.9590.100.1.2.406725628213826290127...,1.3.6.1.4.1.9590.100.1.2.281397494612871934937...,1.3.6.1.4.1.9590.100.1.2.281397494612871934937...


In [ ]:
# --- Step 3: Save cleaned CSV ---
df.to_csv("cleaned_calc_description_train.csv", index=False)

print("Done! Cleaned CSV saved as cleaned_calc_description_train.csv")

Done! Cleaned CSV saved as cleaned_calc_description_train_only.csv
